In [1]:
import pandas as pd

# Load the Excel file
xlsx_file = pd.read_excel('HEAT_Tables_0517_am.xlsx', sheet_name=['RP1', 'Abj_outputs', 'Jhb_outputs'])

# Extract each DataFrame from the dictionary
df_rp1 = xlsx_file['RP1']
df_abj = xlsx_file['Abj_outputs']
df_jhb = xlsx_file['Jhb_outputs']


In [2]:
# Function to map months and years to columns
def map_month_year(df, start_year=2023):
    month_map = {}
    encountered_dec = False
    months_in_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for column in df.columns:
        if any(month in column for month in months_in_order):
            base_month_name = ''.join(filter(str.isalpha, column))
            if base_month_name == 'Jan' and encountered_dec:
                start_year += 1
            if base_month_name == 'Dec':
                encountered_dec = True
            month_map[column] = f'{base_month_name} {start_year}'
    return df.rename(columns=month_map)

# Apply the mapping function to each DataFrame
df_rp1 = map_month_year(df_rp1)
df_abj = map_month_year(df_abj)
df_jhb = map_month_year(df_jhb)


In [3]:
# Convert column names to datetime
def convert_column_to_datetime(df):
    new_columns = []
    for col in df.columns:
        if col == 'Stage':
            new_columns.append(col)
        else:
            date_str = col.strip() + ' 1'
            new_columns.append(pd.to_datetime(date_str, format='%b %Y %d', errors='coerce'))
    df.columns = new_columns

# Apply the conversion to each DataFrame
for dataframe in [df_rp1, df_abj, df_jhb]:
    convert_column_to_datetime(dataframe)


In [4]:
# Define the stage order
stage_order = [
    'Contact procedures not initiated',
    '1st or 2nd invites',
    '3rd or more invites',
    'Data sharing discussions and eligibility check',
    'DTA in progress',
    'DTA completed',
    'Data sets in hand',
    'Database harmonization',
    'Ineligible/declined participation/data currently unavailable'
]

# Reindex each DataFrame to ensure consistent stage order
df_rp1 = df_rp1.set_index('Stage').reindex(stage_order).reset_index()
df_abj = df_abj.set_index('Stage').reindex(stage_order).reset_index()
df_jhb = df_jhb.set_index('Stage').reindex(stage_order).reset_index()


In [5]:
# Define the stage order
stage_order = [
    'Contact procedures not initiated',
    '1st or 2nd invites',
    '3rd or more invites',
    'Data sharing discussions and eligibility check',
    'DTA in progress',
    'DTA completed',
    'Data sets in hand',
    'Database harmonization',
    'Ineligible/declined participation/data currently unavailable'
]

# Reindex each DataFrame to ensure consistent stage order
df_rp1 = df_rp1.set_index('Stage').reindex(stage_order).reset_index()
df_abj = df_abj.set_index('Stage').reindex(stage_order).reset_index()
df_jhb = df_jhb.set_index('Stage').reindex(stage_order).reset_index()


In [6]:
# Fill NaN with 0
df_rp1 = df_rp1.fillna(0)
df_abj = df_abj.fillna(0)
df_jhb = df_jhb.fillna(0)


In [8]:
# Exclude the 'Stage' column and get all date columns
date_columns_rp1 = set(df_rp1.columns[1:])
date_columns_abj = set(df_abj.columns[1:])
date_columns_jhb = set(df_jhb.columns[1:])

# Get the union of all date columns
all_date_columns = sorted(date_columns_rp1.union(date_columns_abj).union(date_columns_jhb))



In [9]:
# Define the full set of columns for reindexing
full_columns = ['Stage'] + all_date_columns

# Reindex and fill missing values with zeros
df_rp1 = df_rp1.set_index('Stage').reindex(columns=all_date_columns).reset_index().fillna(0)
df_abj = df_abj.set_index('Stage').reindex(columns=all_date_columns).reset_index().fillna(0)
df_jhb = df_jhb.set_index('Stage').reindex(columns=all_date_columns).reset_index().fillna(0)


In [10]:
# Sum the DataFrames
df_combined = df_rp1.copy()
df_combined[all_date_columns] = (
    df_rp1[all_date_columns].values +
    df_abj[all_date_columns].values +
    df_jhb[all_date_columns].values
)


In [11]:
# Print the combined DataFrame for verification
print(df_combined)


                                               Stage  2023-01-01 00:00:00  \
0                   Contact procedures not initiated                110.0   
1                                 1st or 2nd invites                 21.0   
2                                3rd or more invites                 29.0   
3     Data sharing discussions and eligibility check                 43.0   
4                                    DTA in progress                 19.0   
5                                      DTA completed                  0.0   
6                                  Data sets in hand                  0.0   
7                             Database harmonization                  0.0   
8  Ineligible/declined participation/data current...                  0.0   

   2023-02-01 00:00:00  2023-03-01 00:00:00  2023-04-01 00:00:00  \
0                109.0                101.0                136.0   
1                 17.0                 22.0                 28.0   
2                 34.0   

In [12]:
# Define the stage order
stage_order = [
    'Contact procedures not initiated',
    '1st or 2nd invites',
    '3rd or more invites',
    'Data sharing discussions and eligibility check',
    'DTA in progress',
    'DTA completed',
    'Data sets in hand',
    'Database harmonization',
    'Ineligible/declined participation/data currently unavailable'
]

# Identify all unique date columns
date_columns_rp1 = set(df_rp1.columns[1:])
date_columns_abj = set(df_abj.columns[1:])
date_columns_jhb = set(df_jhb.columns[1:])

# Get the union of all date columns
all_date_columns = sorted(date_columns_rp1.union(date_columns_abj).union(date_columns_jhb))

# Define the full set of columns for reindexing
full_columns = ['Stage'] + all_date_columns

# Reindex each DataFrame to ensure consistent stage order and columns
df_rp1 = df_rp1.set_index('Stage').reindex(index=stage_order, columns=all_date_columns).reset_index().fillna(0)
df_abj = df_abj.set_index('Stage').reindex(index=stage_order, columns=all_date_columns).reset_index().fillna(0)
df_jhb = df_jhb.set_index('Stage').reindex(index=stage_order, columns=all_date_columns).reset_index().fillna(0)

# Sum the DataFrames
df_combined = df_rp1.copy()
df_combined[all_date_columns] = (
    df_rp1[all_date_columns].values +
    df_abj[all_date_columns].values +
    df_jhb[all_date_columns].values
)

# Print the combined DataFrame for verification
print(df_combined)


                                               Stage  2023-01-01 00:00:00  \
0                   Contact procedures not initiated                110.0   
1                                 1st or 2nd invites                 21.0   
2                                3rd or more invites                 29.0   
3     Data sharing discussions and eligibility check                 43.0   
4                                    DTA in progress                 19.0   
5                                      DTA completed                  0.0   
6                                  Data sets in hand                  0.0   
7                             Database harmonization                  0.0   
8  Ineligible/declined participation/data current...                  0.0   

   2023-02-01 00:00:00  2023-03-01 00:00:00  2023-04-01 00:00:00  \
0                109.0                101.0                136.0   
1                 17.0                 22.0                 28.0   
2                 34.0   

In [13]:
# Prepare data for the latest month
latest_month = df_combined.columns[-1]
data_latest = df_combined[['Stage', latest_month]].copy()
data_latest.columns = ['Stage', 'Count']

# Remove stages that are not needed
data_latest = data_latest[~data_latest['Stage'].str.contains("Total|Ineligible")]

# Map stages to indices
stage_labels = data_latest['Stage'].tolist()
label_indices = list(range(len(stage_labels)))
label_mapping = dict(zip(stage_labels, label_indices))

# Define the data acquisition order explicitly (excluding 'Ineligible/declined...')
data_acquisition_order = [stage for stage in stage_order if stage != 'Ineligible/declined participation/data currently unavailable']


In [14]:
print("df_rp1 shape:", df_rp1.shape)
print("df_abj shape:", df_abj.shape)
print("df_jhb shape:", df_jhb.shape)
print("Columns:", df_rp1.columns)


df_rp1 shape: (9, 22)
df_abj shape: (9, 22)
df_jhb shape: (9, 22)
Columns: Index([            'Stage', 2023-01-01 00:00:00, 2023-02-01 00:00:00,
       2023-03-01 00:00:00, 2023-04-01 00:00:00, 2023-05-01 00:00:00,
       2023-06-01 00:00:00, 2023-07-01 00:00:00, 2023-08-01 00:00:00,
       2023-09-01 00:00:00, 2023-10-01 00:00:00, 2023-11-01 00:00:00,
       2023-12-01 00:00:00, 2024-01-01 00:00:00, 2024-02-01 00:00:00,
       2024-03-01 00:00:00, 2024-04-01 00:00:00, 2024-05-01 00:00:00,
       2024-06-01 00:00:00, 2024-07-01 00:00:00, 2024-08-01 00:00:00,
       2024-09-01 00:00:00],
      dtype='object')


In [17]:
# Normalize stage names in data_latest
data_latest['Stage'] = data_latest['Stage'].str.strip()

# Normalize stage names in data_acquisition_order
data_acquisition_order = [stage.strip() for stage in data_acquisition_order]

# Reconstruct label_mapping
label_indices = list(range(len(data_acquisition_order)))
label_mapping = dict(zip(data_acquisition_order, label_indices))
stage_labels = data_acquisition_order  # Ensure labels are in the correct order

# Filter data_latest to only include stages in data_acquisition_order
data_latest = data_latest[data_latest['Stage'].isin(data_acquisition_order)]

# Create source, target, and value lists
source = []
target = []
value = []

for i in range(len(data_acquisition_order) - 1):
    source_stage = data_acquisition_order[i]
    target_stage = data_acquisition_order[i + 1]
    source.append(label_mapping[source_stage])
    target.append(label_mapping[target_stage])
    count_value = data_latest.loc[data_latest['Stage'] == target_stage, 'Count'].values
    if count_value.size > 0:
        value.append(count_value[0])
    else:
        value.append(0)


In [18]:
print("source:", source)
print("target:", target)
print("value:", value)
print("stage_labels:", stage_labels)


source: [0, 1, 2, 3, 4, 5, 6]
target: [1, 2, 3, 4, 5, 6, 7]
value: [3.0, 26.0, 57.0, 62.0, 6.0, 22.0, 21.0]
stage_labels: ['Contact procedures not initiated', '1st or 2nd invites', '3rd or more invites', 'Data sharing discussions and eligibility check', 'DTA in progress', 'DTA completed', 'Data sets in hand', 'Database harmonization']


In [20]:
import pandas as pd
import plotly.graph_objects as go

# Assuming df_combined has been prepared as in previous steps

# Prepare data for the latest month
latest_month = df_combined.columns[-1]
data_latest = df_combined[['Stage', latest_month]].copy()
data_latest.columns = ['Stage', 'Count']

# Remove stages that are not needed
data_latest = data_latest[~data_latest['Stage'].str.contains("Total|Ineligible")]

# Normalize stage names to ensure consistency
data_latest['Stage'] = data_latest['Stage'].str.strip()
data_acquisition_order = [stage.strip() for stage in stage_order if stage != 'Ineligible/declined participation/data currently unavailable']

# Reconstruct label_mapping
label_indices = list(range(len(data_acquisition_order)))
label_mapping = dict(zip(data_acquisition_order, label_indices))
stage_labels = data_acquisition_order  # Ensure labels are in the correct order

# Filter data_latest to only include stages in data_acquisition_order
data_latest = data_latest[data_latest['Stage'].isin(data_acquisition_order)]

# Create source, target, and value lists
source = []
target = []
value = []

for i in range(len(data_acquisition_order) - 1):
    source_stage = data_acquisition_order[i]
    target_stage = data_acquisition_order[i + 1]
    source.append(label_mapping[source_stage])
    target.append(label_mapping[target_stage])
    count_value = data_latest.loc[data_latest['Stage'] == target_stage, 'Count'].values
    if count_value.size > 0:
        value.append(count_value[0])
    else:
        value.append(0)

# Adjust label text sizes if necessary to prevent overlap
node_labels = [label if len(label) <= 30 else label[:27] + '...' for label in stage_labels]

# Create the Sankey diagram
fig_sankey = go.Figure(data=[go.Sankey(
    arrangement='snap',  # Arrange nodes to minimize label overlap
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color="skyblue",
        font=dict(size=12),  # Corrected property
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color="lightblue"
    ))])

fig_sankey.update_layout(
    title_text="Flow of Studies Through Data Acquisition Stages",
    font_size=12,
    height=600  # Adjust height to prevent label overlap
)
fig_sankey.show()


ValueError: Invalid property specified for object of type plotly.graph_objs.sankey.Node: 'font'

Did you mean "line"?

    Valid properties:
        align
            Sets the alignment method used to position the nodes
            along the horizontal axis.
        color
            Sets the `node` color. It can be a single value, or an
            array for specifying color for each `node`. If
            `node.color` is omitted, then the default `Plotly`
            color palette will be cycled through to have a variety
            of colors. These defaults are not fully opaque, to
            allow some visibility of what is beneath the node.
        colorsrc
            Sets the source reference on Chart Studio Cloud for
            `color`.
        customdata
            Assigns extra data to each node.
        customdatasrc
            Sets the source reference on Chart Studio Cloud for
            `customdata`.
        groups
            Groups of nodes. Each group is defined b

In [23]:
help(go.Sankey.Node)


AttributeError: type object 'Sankey' has no attribute 'Node'